# LLM 11: Evaluation (Intro)

Hands-on:
1. Build a tiny labeled eval set
2. Exact-match baseline
3. LLM-as-judge with rubric
4. Pairwise A/B prompt comparison
5. Exercises: position-bias check, CI-style regression eval

Deps: `pip install anthropic openai`

## 1. A Tiny Eval Set

In [ ]:
eval_set = [
    {'input': 'Capital of France?',                  'expected': 'Paris'},
    {'input': 'Capital of Japan?',                   'expected': 'Tokyo'},
    {'input': 'Who wrote Hamlet?',                   'expected': 'William Shakespeare'},
    {'input': 'What is 13 x 7?',                     'expected': '91'},
    {'input': 'Year of the French Revolution start?','expected': '1789'},
    {'input': 'Chemical symbol for gold?',           'expected': 'Au'},
    {'input': 'Largest ocean on Earth?',             'expected': 'Pacific Ocean'},
    {'input': 'Author of 1984?',                     'expected': 'George Orwell'},
]

## 2. Model Caller

In [ ]:
import os

def call_llm(system, user, temperature=0):
    if os.getenv('ANTHROPIC_API_KEY'):
        from anthropic import Anthropic
        r = Anthropic().messages.create(
            model='claude-sonnet-4-6', max_tokens=200, temperature=temperature,
            system=system, messages=[{'role':'user','content':user}])
        return r.content[0].text
    if os.getenv('OPENAI_API_KEY'):
        from openai import OpenAI
        r = OpenAI().chat.completions.create(
            model='gpt-4o-mini', temperature=temperature,
            messages=[{'role':'system','content':system},{'role':'user','content':user}])
        return r.choices[0].message.content
    return '[no provider]'

## 3. Exact-Match Baseline

Rigid but useful as a sanity floor.

In [ ]:
system_terse = 'Answer in as few words as possible. No punctuation, no explanation.'

def exact_match(pred, expected):
    return pred.strip().lower() == expected.strip().lower()

n_correct = 0
for ex in eval_set:
    pred = call_llm(system_terse, ex['input'])
    ok = exact_match(pred, ex['expected'])
    n_correct += int(ok)
    print(f'{"✓" if ok else "✗"}  Q: {ex["input"]:<40}  pred: {pred.strip()[:40]:<40}  exp: {ex["expected"]}')
print(f'\nexact-match accuracy: {n_correct}/{len(eval_set)}')

## 4. LLM-as-Judge with Rubric

A rubric-based judge is less rigid than exact match — it credits semantically-correct answers that don't match lexically.

In [ ]:
import json, re

JUDGE_SYSTEM = '''You are a strict evaluator. Given a question, an expected answer, and a candidate answer,
decide if the candidate is correct. Output only JSON: {"correct": true|false, "reason": "..."}.'''

def judge(question, expected, actual):
    prompt = (f'Question: {question}\n'
              f'Expected: {expected}\n'
              f'Candidate: {actual}\n\n'
              f'Is the candidate correct? Respond with JSON only.')
    raw = call_llm(JUDGE_SYSTEM, prompt)
    try:
        m = re.search(r'\{.*?\}', raw, re.S)
        return json.loads(m.group(0))
    except Exception:
        return {'correct': False, 'reason': f'parse failed: {raw}'}

graded = 0
for ex in eval_set:
    pred = call_llm('Answer the question naturally in one sentence.', ex['input'])
    j = judge(ex['input'], ex['expected'], pred)
    graded += int(j['correct'])
    mark = '✓' if j['correct'] else '✗'
    print(f'{mark}  Q: {ex["input"]:<35}  pred: {pred.strip()[:50]:<50}')
print(f'\nLLM-judge accuracy: {graded}/{len(eval_set)}')

## 5. Pairwise A/B Prompt Comparison

In [ ]:
prompt_A = 'Answer in one sentence.'
prompt_B = 'Answer in one sentence, with a historical note.'

PAIR_JUDGE = '''You are picking the better of two answers to a question.
Return JSON: {"winner": "A" or "B", "reason": "..."}.
Penalize answers that are verbose without adding relevant information.'''

def pairwise(question, ans_a, ans_b):
    prompt = f'Question: {question}\n\nAnswer A: {ans_a}\n\nAnswer B: {ans_b}\n\nWhich is better?'
    raw = call_llm(PAIR_JUDGE, prompt)
    try:
        return json.loads(re.search(r'\{.*?\}', raw, re.S).group(0))
    except Exception:
        return {'winner': '?', 'reason': raw[:100]}

a_wins, b_wins = 0, 0
for ex in eval_set[:4]:
    a = call_llm(prompt_A, ex['input'])
    b = call_llm(prompt_B, ex['input'])
    result = pairwise(ex['input'], a, b)
    if result['winner'] == 'A': a_wins += 1
    elif result['winner'] == 'B': b_wins += 1
    print(f'Q: {ex["input"]:<35}  winner: {result["winner"]}  ({result["reason"][:60]})')
print(f'\nprompt A: {a_wins}   prompt B: {b_wins}')

## 6. Exercise: Position-Bias Check

Repeat the pairwise comparison above but **swap the order** of A and B for each example. Compute A-win rate with A-first vs A-second. A large gap means your judge has position bias; mitigate by always evaluating both orderings and averaging.

## 7. Exercise: CI-Style Prompt Regression

Write a pytest-style harness:
```python
def test_prompt_v2_not_worse_than_v1():
    v1 = run_eval(PROMPT_V1)
    v2 = run_eval(PROMPT_V2)
    assert v2.accuracy >= v1.accuracy - 0.05  # allow 5pp wiggle
```

Wire it into CI. Any prompt change that drops quality measurably fails the build. This is the single most impactful eval practice for teams shipping LLM features.

## 8. Takeaways

- **Evaluation is measurement, not philosophy.** Build the set; run it.
- **Exact match is a floor**; LLM-as-judge handles open-ended answers.
- **Pairwise preference** is high-signal for picking between prompts/models.
- **Bias-check your judge** — position, verbosity, self-preference.
- **Regress in CI.** A prompt change without a measurement is a guess.

Next: **12 — Token Economics**, where we turn all these knobs into dollars.